# Construction d'une bibliothèque de tarification actuarielle réutilisable avec PROC FCMP


## Synthèse

Un assureur IARD encapsule ses calculs de tarification de base — prime pure, chargement de frais/risque, pondération de crédibilité à fluctuation limitée, et estimation actualisée des réserves — sous forme de fonctions personnalisées et d'une sous-routine à sorties multiples dans **PROC FCMP**. La bibliothèque compilée est enregistrée via l'option système `CMPLIB=`, puis appelée ligne par ligne depuis une étape DATA qui tarife un portefeuille synthétique de 100 polices. Chaque valeur tarifée de ce notebook — le facteur de crédibilité `z`, la prime pure pondérée, la prime chargée et la réserve de sinistre actualisée — est produite par ces routines compilées, et non par du calcul en ligne. Résultat : le ratio de sinistralité implicite atteint 60,5 % (rural), 55,8 % (périurbain) et 63,6 % (urbain) — confortablement sous 100 % dans chaque segment, confirmant que la prime chargée couvre la perte attendue tandis que l'étape de tarification reste propre et auditable.


## Sources de données

| Jeu de données | Lignes | Description | Variables clés |
|---------|------|-------------|---------------|
| `work.portfolio` | 100 | Portefeuille synthétique de polices IARD en vigueur, généré en ligne avec `rand()` | `policy_id`, `region` (urbain/périurbain/rural), `years_insured`, `exposure` (véhicule-années), `claim_count` (Poisson), `incurred_loss` (sévérité gamma x fréquence), `class_pure_prem` (tarif manuel par région) |

L'étape DATA parcourt une plage `policy_id` plus large, mais cet environnement s'exécute sans licence ; la sortie est donc plafonnée aux **100 premières observations** — le livre tarifé ci-dessous porte sur ces 100 polices.


# Construction d'une bibliothèque de tarification actuarielle réutilisable avec PROC FCMP

Les actuaires répètent les mêmes calculs dans les travaux de tarification, de provisionnement et de reporting : convertir les pertes en *prime pure*, appliquer des *chargements de frais et de risque* pour obtenir une prime chargée, pondérer l'expérience propre d'une police avec un tarif de classe via la *crédibilité*, et *actualiser* les flux de trésorerie futurs à leur valeur actuelle. Retaper ces formules dans chaque étape DATA favorise les erreurs de copier-coller et complique les changements d'hypothèses.

**PROC FCMP** (le compilateur de fonctions SAS) permet de définir chaque formule une seule fois sous forme de fonction ou de sous-routine nommée, de stocker les routines compilées dans une bibliothèque, puis de les appeler comme n'importe quelle fonction SAS intégrée. Dans ce notebook, nous allons :

1. Compiler une petite bibliothèque de fonctions actuarielles avec `PROC FCMP`.
2. L'enregistrer pour la session avec l'option système `CMPLIB=`.
3. Générer un portefeuille IARD synthétique.
4. Tarifer chaque police en appelant nos fonctions personnalisées et une sous-routine à sorties multiples depuis une seule étape DATA.
5. Synthétiser et interpréter le portefeuille tarifé.


## Étape 1 — Générer un portefeuille de polices synthétique

Nous simulons un livre de polices auto en vigueur (les 100 premières sont tarifées ci-dessous sous le plafond du mode sans licence). Chaque police appartient à une région de tarification avec sa propre *prime pure* manuelle (perte attendue par véhicule-année). Le nombre de sinistres suit un processus de Poisson mis à l'échelle par l'exposition, et les sévérités suivent une distribution gamma, de sorte que `incurred_loss` est une perte composée réaliste (Poisson x gamma). `years_insured` déterminera ensuite le poids de crédibilité. Une graine fixe via `call streaminit` rend les données reproductibles.


In [1]:
DONNÉES portfolio;
    APPELER streaminit(20260531);
    LONGUEUR region $15;
    FAIRE policy_id = 10001 JUSQU_À 12000;
        /* Attribution d'une région de tarification */
        u = rand('uniform');
        SI u < 0.40 ALORS FAIRE; region = 'urbain';    base_pp = 820; lambda = 0.18; FIN;
        SINON SI u < 0.75 ALORS FAIRE; region = 'périurbain'; base_pp = 540; lambda = 0.11; FIN;
        SINON FAIRE; region = 'rural';    base_pp = 360; lambda = 0.07; FIN;

        /* Ancienneté (années assurées) et exposition (véhicule-années) */
        years_insured = 1 + rand('poisson', 3);
        exposure = round(0.5 + rand('gamma', 4) / 4, 0.01);

        /* Processus composé de sinistres : fréquence Poisson, sévérité gamma */
        claim_count = rand('poisson', lambda * exposure);
        incurred_loss = 0;
        FAIRE c = 1 JUSQU_À claim_count;
            incurred_loss = incurred_loss + rand('gamma', 2.0) * 2500;
        FIN;
        incurred_loss = round(incurred_loss, 0.01);

        /* Prime pure de classe manuelle pour l'exposition de cette police */
        class_pure_prem = round(base_pp * exposure, 0.01);

        SORTIE;
    FIN;
    GARDER policy_id region years_insured exposure claim_count
         incurred_loss class_pure_prem;
EXÉCUTER;

PROCÉDURE IMPRIMER DONNÉES=portfolio(obs=8) noobs;
    ÉTIQUETTE policy_id="Identifiant de police" region="Région" years_insured="Ancienneté (années)"
        exposure="Exposition (véhicule-années)" claim_count="Nombre de sinistres"
        incurred_loss="Perte encourue" class_pure_prem="Prime pure de classe";
    TITRE 'Les 8 premières polices simulées';
EXÉCUTER;


                                            Les 8 premières polices simulées                                            

Identifiant de police       Région    Ancienneté (années)    Exposition (véhicule-années)  Nombre de sinistres  Perte encourue  Prime pure de classe
                10001  rural                            5                            1.36                    0               0                 489.6
                10002  urbain                           8                            0.97                    1         3432.28                 795.4
                10003  urbain                           2                            1.53                    2         7155.92                1254.6
                10004  périurbain                       9                             2.4                    0               0                  1296
                10005  rural                            5                            0.99                    0               0       


NOTE: DATA portfolio

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote portfolio (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.48 seconds
  cpu   0.48 seconds
NOTE: PROC PRINT data=portfolio

NOTE: PROC PRINT completed: 8 observations printed, 7 variables


## Étape 2 — Compiler la bibliothèque de fonctions actuarielles

Voici maintenant le cœur du notebook. `PROC FCMP` avec `OUTLIB=work.actfuncs.pricing` compile quatre fonctions et une sous-routine dans le paquet `pricing` du jeu de données `work.actfuncs` :

- **`pure_premium`** — perte observée par unité d'exposition (fréquence x sévérité combinées).
- **`credibility_z`** — facteur de crédibilité à fluctuation limitée `Z = sqrt(n / (n + k))`, où `n` est le nombre d'années-exposition de la police et `k` une constante de calibrage.
- **`charged_premium`** — applique un chargement de risque proportionnel et un taux de frais fixe à un coût de perte pour obtenir la prime réellement chargée.
- **`pv_reserve`** — valeur actuelle d'un règlement de sinistre futur, `FV / (1+r)**t`, utilisée pour actualiser les réserves de dossier.
- **`blend_premium`** (une SOUS-ROUTINE) — utilise `OUTARGS` pour renvoyer *deux* valeurs en une fois : la prime pure pondérée par la crédibilité et le facteur de crédibilité utilisé, afin que l'étape DATA obtienne les deux en un seul appel.

Chaque routine se termine par `ENDSUB`, et la sous-routine nomme ses arguments modifiables avec `OUTARGS`.


In [2]:
PROCÉDURE fcmp outlib=work.actfuncs.pricing;

    /* Prime pure observée : coût de perte par unité d'exposition */
    function pure_premium(loss, exposure);
        SI exposure <= 0 ALORS RETURN(.);
        RETURN(loss / exposure);
    endsub;

    /* Crédibilité à fluctuation limitée : Z = sqrt(n / (n + k)) */
    function credibility_z(n_years, k);
        SI n_years <= 0 ALORS RETURN(0);
        RETURN(sqrt(n_years / (n_years + k)));
    endsub;

    /* Prime chargée = coût de perte majoré du chargement de risque + frais */
    function charged_premium(loss_cost, risk_load, expense_ratio);
        gross = loss_cost * (1 + risk_load);
        SI expense_ratio >= 1 ALORS RETURN(.);
        RETURN(gross / (1 - expense_ratio));
    endsub;

    /* Valeur actuelle d'un règlement de sinistre futur actualisé sur t années au taux r */
    function pv_reserve(future_value, r, t);
        RETURN(future_value / (1 + r) ** t);
    endsub;

    /* Pondération de crédibilité : renvoie la prime pure pondérée ET le Z utilisé */
    subroutine blend_premium(own_pp, class_pp, n_years, k, blended, z_used);
        outargs blended, z_used;
        z_used = credibility_z(n_years, k);
        blended = z_used * own_pp + (1 - z_used) * class_pp;
    endsub;

EXÉCUTER;



               The FCMP Procedure

  Output Library: work.actfuncs.pricing
  User-defined Function: pure_premium
  User-defined Function: credibility_z
  User-defined Function: charged_premium
  User-defined Function: pv_reserve
  User-defined Subroutine: blend_premium




NOTE: PROC FCMP outlib=work.actfuncs.pricing

NOTE: Function pure_premium stored to work.actfuncs.pricing.
NOTE: Function credibility_z stored to work.actfuncs.pricing.
NOTE: Function charged_premium stored to work.actfuncs.pricing.
NOTE: Function pv_reserve stored to work.actfuncs.pricing.
NOTE: Subroutine blend_premium stored to work.actfuncs.pricing.


## Étape 3 — Enregistrer la bibliothèque avec CMPLIB=

Compiler les routines ne suffit pas ; SAS doit savoir où les trouver lorsqu'une étape DATA ou une procédure ultérieure référence un nom qu'elle ne reconnaît pas comme intégré. L'option système `CMPLIB=` pointe vers le jeu de données (et non le nom de paquet à trois niveaux) contenant le code compilé. Après cette instruction `OPTIONS`, chaque fonction et sous-routine ci-dessus devient appelable par son nom.


In [3]:
options cmplib=work.actfuncs;



NOTE: Option CMPLIB changed to WORK.ACTFUNCS.


## Étape 4 — Tarifer le portefeuille avec les fonctions personnalisées

L'étape DATA de tarification est maintenant presque dénuée de calcul en ligne — l'intention actuarielle se lit directement dans les noms de fonctions. Pour chaque police, nous :

1. Calculons la prime pure observée propre à la police avec `pure_premium`.
2. Appelons la sous-routine `blend_premium` pour la pondérer par crédibilité contre le tarif de classe régional, en récupérant à la fois le coût de perte pondéré et le facteur de crédibilité via `OUTARGS`.
3. Majorons le coût de perte pondéré pour obtenir une prime chargée avec un chargement de risque de 12 % et un taux de frais de 25 % via `charged_premium`.
4. Estimons la réserve de dossier encore ouverte à 35 % de la perte encourue de la police et l'actualisons sur trois ans à 4 % avec `pv_reserve`.

Notez que les arguments de sortie de la sous-routine (`blended_pp`, `z`) doivent être initialisés avant le `CALL`. La VA de la réserve varie d'une police à l'autre car elle dépend de la perte encourue réelle de chaque police — les polices sans sinistre portent une réserve nulle, donc leur `reserve_pv` est également nul.


In [4]:
DONNÉES rated;
    DÉFINIR portfolio;

    /* 1. Expérience de perte propre de la police, sous forme de prime pure */
    own_pp = round(pure_premium(incurred_loss, exposure), 0.01);

    /* 2. Pondération par crédibilité de l'expérience propre contre le tarif de classe.
          k = 4 années-exposition pour une crédibilité quasi complète. */
    blended_pp = .;
    z = .;
    APPELER blend_premium(own_pp, class_pure_prem / exposure,
                       years_insured, 4, blended_pp, z);
    blended_pp = round(blended_pp, 0.01);
    z = round(z, 0.001);

    /* 3. Conversion du coût de perte pondéré (par véhicule-année) en prime chargée */
    loss_cost = blended_pp * exposure;
    premium = round(charged_premium(loss_cost, 0.12, 0.25), 0.01);

    /* 4. Réserve de dossier en cours = 35 % de la perte encourue, réglée dans
          3 ans ; on l'actualise à 4 %. */
    case_reserve = round(0.35 * incurred_loss, 0.01);
    reserve_pv   = round(pv_reserve(case_reserve, 0.04, 3), 0.01);

    GARDER policy_id region years_insured exposure incurred_loss
         own_pp class_pure_prem blended_pp z premium
         case_reserve reserve_pv;
EXÉCUTER;

PROCÉDURE IMPRIMER DONNÉES=rated(obs=10) noobs;
    ÉTIQUETTE policy_id="Identifiant de police" region="Région" years_insured="Ancienneté (années)"
        exposure="Exposition (véhicule-années)" own_pp="Prime pure propre"
        blended_pp="Prime pure pondérée" z="Facteur de crédibilité (z)"
        premium="Prime chargée" reserve_pv="VA de la réserve";
    TITRE 'Portefeuille tarifé (10 premières polices)';
    VAR policy_id region years_insured exposure own_pp
        blended_pp z premium reserve_pv;
EXÉCUTER;


                                       Portefeuille tarifé (10 premières polices)                                       

Identifiant de police       Région    Ancienneté (années)    Exposition (véhicule-années)  Prime pure propre    Prime pure pondérée    Facteur de crédibilité (z)   Prime chargée   VA de la réserve
                10001  rural                            5                            1.36                  0                  91.67                         0.745          186.18                  0
                10002  urbain                           8                            0.97            3538.43                3039.59                         0.816         4402.95            1067.95
                10003  urbain                           2                            1.53            4677.07                3046.88                         0.577         6961.51            2226.55
                10004  périurbain                       9                             2.4 


NOTE: DATA rated


NOTE: Read 100 rows from portfolio.
NOTE: Wrote rated (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.07 seconds
  cpu   0.07 seconds
NOTE: PROC PRINT data=rated

NOTE: PROC PRINT completed: 10 observations printed, 9 variables


## Étape 5 — Synthétiser le portefeuille tarifé

Chaque police étant tarifée via la même bibliothèque compilée, nous pouvons agréger le portefeuille par région. Nous rapportons la prime chargée moyenne, le facteur de crédibilité moyen, la perte encourue totale et la réserve de dossier totale actualisée, puis nous en déduisons le ratio de sinistralité implicite par segment afin de voir si la prime chargée couvre la perte attendue.


In [5]:
PROCÉDURE MOYENNES DONNÉES=rated mean sum maxdec=2 nonobs;
    CLASSE region;
    VAR premium z incurred_loss reserve_pv;
    ÉTIQUETTE region="Région" premium="Prime chargée" z="Facteur de crédibilité (z)"
        incurred_loss="Perte encourue" reserve_pv="VA de la réserve";
    TITRE "Synthèse du portefeuille par région de tarification";
EXÉCUTER;

/* Ratio de sinistralité implicite = perte encourue / prime chargée, plus la
   réserve actualisée portée par le segment, par région. */
PROCÉDURE sql;
    TITRE "Ratio de sinistralité implicite et VA de la réserve par région";
    SÉLECTIONNER region,
           count(*)                              COMME n_policies label="Nombre de polices",
           sum(incurred_loss)                    COMME total_loss     label="Perte totale" format=dollar12.2,
           sum(premium)                          COMME total_premium  label="Prime totale" format=dollar12.2,
           sum(incurred_loss) / sum(premium)     COMME loss_ratio     label="Ratio de sinistralité" format=percent8.1,
           sum(reserve_pv)                       COMME total_pv_reserve label="VA totale de la réserve" format=dollar12.2
    DEPUIS rated
    GROUPE PAR region
    ORDRE PAR region;
QUIT;
TITRE;


                                  Synthèse du portefeuille par région de tarification                                   

                                                  The MEANS Procedure

                                       Analysis Variable : premium Prime chargée

        Région                Mean            Sum
        -----------------------------------------
        périurbain          813.04       34147.74
        rural               476.61       11438.62
        urbain             1987.17       67563.93
        -----------------------------------------

                                   Analysis Variable : z Facteur de crédibilité (z)

        Région                Mean            Sum
        -----------------------------------------
        périurbain            0.68          28.36
        rural                 0.71          17.14
        urbain                0.70          23.90
        -----------------------------------------

                                    An


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC SQL 

NOTE: PROC SQL statement used.


## Interprétation des résultats

L'étape DATA de tarification n'explicite jamais une seule formule d'actualisation ou de crédibilité en ligne — elle appelle simplement `pure_premium`, `blend_premium`, `charged_premium` et `pv_reserve`. C'est là tout l'intérêt de **PROC FCMP** : les hypothèses actuarielles vivent dans une seule bibliothèque compilée qui peut être testée unitairement, versionnée et réutilisée dans les travaux de tarification, de provisionnement et de reporting. Changer la constante de crédibilité `k`, le chargement de risque ou le taux de frais ne demande qu'une seule modification dans la bibliothèque, et non une recherche dans chaque programme.

En lisant le livre tarifé et l'agrégation régionale :

- **La crédibilité (`z`)** augmente avec `years_insured`, exactement comme le dicte la formule à fluctuation limitée `Z = sqrt(n / (n + k))`. Parmi les dix premières polices, la police périurbaine d'un an (10006) obtient `z = 0,447`, la police urbaine de deux ans (10003) `z = 0,577`, tandis que la police périurbaine de neuf ans (10004) atteint `z = 0,832`. Les polices à expérience mince sont ramenées vers le tarif de classe régional ; les polices de longue ancienneté s'appuient sur leurs propres pertes.
- **La pondération en action :** les polices sans sinistre (la majorité du livre) ont `own_pp = 0`, donc `blend_premium` renvoie une `blended_pp` proche de `(1 - z)` fois le tarif de classe — par exemple la police 10001 (rurale, `z = 0,745`) atterrit à `91,67` contre un tarif de classe de `360`/véhicule-année. Les deux polices urbaines ayant effectivement subi des sinistres, 10002 et 10003, tirent au contraire leur coût de perte pondéré vers leur propre expérience élevée (`3039,59` et `3046,88`).
- **La prime chargée** se situe au-dessus du coût de perte pondéré car `charged_premium` ajoute un chargement de risque de 12 % et majore pour un taux de frais de 25 %, un multiplicateur fixe de `1,12 / 0,75 = 1,493` sur le coût de perte. La prime chargée moyenne s'élève à `476,61` (rural), `813,04` (périurbain) et `1 987,17` (urbain).
- **Les réserves actualisées :** `pv_reserve` actualise la réserve de dossier en cours de chaque police (35 % de la perte encourue) sur trois ans à 4 %, un facteur d'actualisation de `0,889`. Les polices sans sinistre portent `reserve_pv = 0` ; les deux sinistrés urbains contribuent `1 067,95` et `2 226,55`. Au global, le livre porte une réserve actualisée de `2 151,56 $` (rural), `5 932,52 $` (périurbain) et `13 364,13 $` (urbain).
- **Les ratios de sinistralité implicites** atteignent 60,5 % (rural), 55,8 % (périurbain) et 63,6 % (urbain) — tous confortablement sous 100 %, donc la prime chargée couvre la perte attendue dans chaque segment. Le segment urbain est le plus élevé, tiré par ses deux grosses pertes simulées ; une véritable revue tarifaire testerait si ce signal persiste sur davantage d'années d'accident avant d'ajuster le tarif manuel.

La sous-routine `blend_premium` illustre également l'idiome `OUTARGS` pour renvoyer plusieurs résultats en un seul `CALL` — le coût de perte pondéré et le facteur de crédibilité `z` reviennent ensemble — ce qui évite des appels de fonction séparés et garde la logique de tarification par police compacte et auditable.
